# Guidance: Constants + cached sensors on the EXISTING linear model (issue #3, step 1)

Goal: validate the two structural changes issue #3 needs -- (1) targeted parameters as
`dolfinx.fem.Constant` instead of baked-in Python floats, and (2) a sensor lookup built
once instead of rebuilt every call -- **before** wiring anything to SciPy, and
**without** depending on the non-linear radiation work from issue #2 (that lives on its
own branch; this one only needs the linear model that is already on `main`).

Do this on the linear model on purpose: if something is wrong here, you want to know
immediately, not after also debugging a Newton solver and a SciPy optimizer at the
same time.


In [ ]:
import time
import numpy as np
import ufl
from mpi4py import MPI
from petsc4py import PETSc
from dolfinx import mesh, fem
import dolfinx.fem.petsc
import dolfinx.geometry as geometry


## Part A -- `dolfinx.fem.Constant` instead of baked-in floats

Today, `advection_coeff` (and every other material parameter) is a plain Python
`float` substituted directly into `a_ufl` when the string is built. That means
changing it requires rebuilding `a_ufl`, recompiling it with `fem.form(...)`, and
reassembling `A` -- exactly the recompilation cost issue #3 wants to avoid across
SciPy iterations.

`dolfinx.fem.Constant` fixes this: it is a UFL symbol that gets compiled ONCE into the
form, and its numerical value can be changed afterwards via `.value = ...` with **no**
recompilation.


In [ ]:
width, thickness = 1.0, 0.5
nx, ny = 60, 20

domain = mesh.create_rectangle(
    MPI.COMM_WORLD,
    [np.array([0.0, 0.0]), np.array([width, thickness])],
    [nx, ny],
    cell_type=mesh.CellType.triangle,
)
V = fem.functionspace(domain, ("CG", 1))

def boundary_bottom(x):
    return np.isclose(x[1], 0.0)

dofs_bottom = fem.locate_dofs_geometrical(V, boundary_bottom)
bcs = [fem.dirichletbc(PETSc.ScalarType(0.0), dofs_bottom, V)]

r_weight = fem.Function(V)
r_weight.interpolate(lambda x: np.maximum(np.abs(x[0]), 1e-14))

uh = fem.Function(V)
u_n = fem.Function(V)
du = ufl.TrialFunction(V)
v = ufl.TestFunction(V)

def grad_cyl(w):
    return ufl.as_vector([ufl.Dx(w, 0), ufl.Dx(w, 1)])

dt = 5.0e-3

# --- targeted parameters as Constants (this is the actual change) ---
thermal_capacity = fem.Constant(domain, PETSc.ScalarType(1.0))
diffusion_coeff = fem.Constant(domain, PETSc.ScalarType(1.0))
advection_coeff = fem.Constant(domain, PETSc.ScalarType(1.0))  # <- the one we'll identify

a_ufl = (
    thermal_capacity * (1.0 / dt) * du * v * r_weight * ufl.dx
    + diffusion_coeff * ufl.dot(grad_cyl(du), grad_cyl(v)) * r_weight * ufl.dx
    + advection_coeff * du * v * r_weight * ufl.ds
)
L_ufl = thermal_capacity * (1.0 / dt) * u_n * v * r_weight * ufl.dx

a = fem.form(a_ufl)    # compiled ONCE
L = fem.form(L_ufl)    # compiled ONCE

def run_forward(h_value: float, n_steps: int = 20) -> float:
    """Run a few steps and return the mean temperature -- NO recompilation."""
    advection_coeff.value = h_value   # <- update, not rebuild
    u_n.x.array[:] = 5.0
    A = dolfinx.fem.petsc.assemble_matrix(a, bcs=bcs)
    A.assemble()
    ksp = PETSc.KSP().create(domain.comm)
    ksp.setOperators(A)
    ksp.setType("preonly")
    ksp.getPC().setType("lu")
    for _ in range(n_steps):
        b = dolfinx.fem.petsc.assemble_vector(L)
        dolfinx.fem.petsc.apply_lifting(b, [a], [bcs])
        b.ghostUpdate(addv=PETSc.InsertMode.ADD_VALUES, mode=PETSc.ScatterMode.REVERSE)
        dolfinx.fem.petsc.set_bc(b, bcs)
        ksp.solve(b, uh.x.petsc_vec)
        uh.x.scatter_forward()
        u_n.x.array[:] = uh.x.array
    return float(uh.x.array.mean())

print("h=1.0 ->", run_forward(1.0))
print("h=5.0 ->", run_forward(5.0))
print("h=1.0 again ->", run_forward(1.0))  # sanity: must match the first run exactly


**Sanity check to run yourself:** the matrix `A` above is still reassembled every call
(its entries change with `advection_coeff.value`, `a` itself does not) -- what did NOT
happen is a call to `fem.form(a_ufl)` again. That single line is the expensive one
(FFCX compiles UFL to C and compiles the C); confirm this by timing `fem.form(a_ufl)`
once vs calling `run_forward` 20 times in a loop -- the per-call cost should be
dominated by assembly + linear solve, not compilation.


## Part B -- cached sensor lookup vs. the naive per-call approach

The legacy code (see the "Extract temperature history" cell in the reference notebook)
rebuilds `dolfinx.geometry.bb_tree` and re-locates every sensor's cell **inside** the
time loop, on every call to `solve()`. Compare that to building the lookup once.


In [ ]:
sensors_rz = np.array([[0.1, 0.25], [0.5, 0.25], [0.9, 0.25]])
points3 = np.zeros((sensors_rz.shape[0], 3))
points3[:, :2] = sensors_rz

def naive_eval(uh):
    """What the legacy code does: rebuild the tree every call."""
    bb = geometry.bb_tree(domain, domain.topology.dim)
    cand = geometry.compute_collisions_points(bb, points3)
    coll = geometry.compute_colliding_cells(domain, cand, points3)
    cells = np.array([coll.links(i)[0] if len(coll.links(i)) else -1 for i in range(len(points3))])
    return uh.eval(points3[cells >= 0], cells[cells >= 0])

# Build once (this is what surroptim.inverse.SensorLocator will formalise):
bb_cached = geometry.bb_tree(domain, domain.topology.dim)
cand_cached = geometry.compute_collisions_points(bb_cached, points3)
coll_cached = geometry.compute_colliding_cells(domain, cand_cached, points3)
cells_cached = np.array([coll_cached.links(i)[0] if len(coll_cached.links(i)) else -1 for i in range(len(points3))])

def cached_eval(uh):
    ok = cells_cached >= 0
    return uh.eval(points3[ok], cells_cached[ok])

n_repeats = 200
t0 = time.perf_counter()
for _ in range(n_repeats):
    naive_eval(uh)
t_naive = time.perf_counter() - t0

t0 = time.perf_counter()
for _ in range(n_repeats):
    cached_eval(uh)
t_cached = time.perf_counter() - t0

print(f"naive:  {t_naive:.4f}s for {n_repeats} calls")
print(f"cached: {t_cached:.4f}s for {n_repeats} calls")
print(f"speedup: {t_naive / t_cached:.1f}x")


If the speedup printed above is not clearly > 1, something is wrong with the caching
(or the mesh is too small for the difference to show at this repeat count) -- increase
`n_repeats` or `nx, ny` before trusting the result. Once this is convincing, move to
`notebooks/main_inverse_problem.ipynb`, which wraps both of these behind
`surroptim.inverse.SensorLocator` and `dolfinx.fem.Constant` inside a proper
`forward_model(theta)` for SciPy.
